In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

print("Google Drive mounted successfully.")

Mounted at /content/drive
Google Drive mounted successfully.


In [ ]:
PROJECT_PATH = "/content/drive/MyDrive/AML_Project"

DATASET_PATH = os.path.join(PROJECT_PATH, "dataset")
OUTPUT_PATH = os.path.join(PROJECT_PATH, "outputs")

FORENSIC_OUTPUT_PATH = os.path.join(
    OUTPUT_PATH,
    "forensic"
)

# Create forensic output directory
os.makedirs(FORENSIC_OUTPUT_PATH, exist_ok=True)

print("Project Path:", PROJECT_PATH)
print("Dataset Path:", DATASET_PATH)
print("Output Path:", OUTPUT_PATH)
print("Forensic Output Path:", FORENSIC_OUTPUT_PATH)

Project Path: /content/drive/MyDrive/AML_Project
Dataset Path: /content/drive/MyDrive/AML_Project/dataset
Output Path: /content/drive/MyDrive/AML_Project/outputs
Forensic Output Path: /content/drive/MyDrive/AML_Project/outputs/forensic


In [ ]:
TRANSACTION_FILE = os.path.join(
    OUTPUT_PATH,
    "integrated_transaction_risk_results.csv"
)

df = pd.read_csv(TRANSACTION_FILE)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (149994, 35)


In [ ]:
print("First 5 rows:")
display(df.head())

First 5 rows:


,transaction_id,sender,receiver,receiver_name,is_merchant,transaction_type,amount,timestamp,atm_id,city,...,sequence_3,sequential_risk_raw,sequential_risk,spatial_risk,integrated_risk_score,risk_category,predicted_fraud,final_prediction,sequential_alert,spatial_alert
0,TX00000001,CUST00001,CUST00840,CUST00840,False,Card,8524.0,2026-01-01 14:08:47,ATM0096,Ahmedabad,...,→ → Card,0.0,0.0,0.0,0.0,Low Risk,0,Normal,0,0
1,TX00021204,CUST00001,MER00169,Indian Oil,True,Fuel,3595.0,2026-01-02 13:31:46,ATM0169,Kochi,...,→ Card → Fuel,0.0,0.0,0.0,0.0,Low Risk,0,Normal,0,0
2,TX00021206,CUST00001,MER00168,Bharat Petroleum,True,Fuel,1275.0,2026-01-02 18:10:34,ATM0084,Bhopal,...,Card → Fuel → Fuel,0.0,0.0,0.0,0.0,Low Risk,0,Normal,0,0
3,TX00021205,CUST00001,CUST02662,CUST02662,False,UPI,3866.0,2026-01-02 22:02:10,ATM0128,Visakhapatnam,...,Fuel → Fuel → UPI,0.0,0.0,0.0,0.0,Low Risk,0,Normal,0,0
4,TX00042517,CUST00001,MER00167,Bharat Petroleum,True,Fuel,921.0,2026-01-03 07:55:52,ATM0097,Hyderabad,...,Fuel → UPI → Fuel,0.0,0.0,0.0,0.0,Low Risk,0,Normal,0,0


In [ ]:
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

print("\nColumns:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i}. {col}")

Number of rows: 149994
Number of columns: 35

Columns:
1. transaction_id
2. sender
3. receiver
4. receiver_name
5. is_merchant
6. transaction_type
7. amount
8. timestamp
9. atm_id
10. city
11. state
12. latitude
13. longitude
14. fraud_type
15. pattern_id
16. is_fraud
17. previous_latitude
18. previous_longitude
19. previous_timestamp
20. time_gap_minutes
21. distance_km
22. rapid_geo_flag
23. previous_transaction_type
24. previous_previous_type
25. sequence_2
26. sequence_3
27. sequential_risk_raw
28. sequential_risk
29. spatial_risk
30. integrated_risk_score
31. risk_category
32. predicted_fraud
33. final_prediction
34. sequential_alert
35. spatial_alert


In [ ]:
profile = pd.DataFrame({
    "Metric": [
        "Total Transactions",
        "Total Customers/Senders",
        "Total Receivers",
        "Total Fraud Transactions",
        "Total Normal Transactions",
        "Total Transaction Types",
        "Total Cities",
        "Minimum Transaction Amount",
        "Maximum Transaction Amount",
        "Average Transaction Amount"
    ],
    "Value": [
        len(df),
        df["sender"].nunique(),
        df["receiver"].nunique(),
        df["is_fraud"].sum(),
        (df["is_fraud"] == 0).sum(),
        df["transaction_type"].nunique(),
        df["city"].nunique(),
        df["amount"].min(),
        df["amount"].max(),
        df["amount"].mean()
    ]
})

display(profile)

,Metric,Value
0,Total Transactions,149994.000000
1,Total Customers/Senders,5000.000000
2,Total Receivers,5353.000000
3,Total Fraud Transactions,1450.000000
4,Total Normal Transactions,148544.000000
5,Total Transaction Types,15.000000
6,Total Cities,21.000000
7,Minimum Transaction Amount,100.000000
8,Maximum Transaction Amount,999431.000000
9,Average Transaction Amount,24103.119973


In [ ]:
profile.to_csv(
    os.path.join(
        FORENSIC_OUTPUT_PATH,
        "forensic_data_profile.csv"
    ),
    index=False
)

print("Saved: forensic_data_profile.csv")

Saved: forensic_data_profile.csv


In [ ]:
missing = pd.DataFrame({
    "column": df.columns,
    "missing_count": df.isnull().sum().values,
    "missing_percentage": (
        df.isnull().sum().values / len(df) * 100
    )
})

missing = missing.sort_values(
    "missing_count",
    ascending=False
)

display(missing.head(15))

,column,missing_count,missing_percentage
14,pattern_id,148544,99.033295
23,previous_previous_type,10000,6.666933
18,previous_timestamp,5000,3.333467
19,time_gap_minutes,5000,3.333467
22,previous_transaction_type,5000,3.333467
17,previous_longitude,5000,3.333467
16,previous_latitude,5000,3.333467
8,atm_id,1450,0.966705
2,receiver,0,0.000000
0,transaction_id,0,0.000000


In [ ]:
missing.to_csv(
    os.path.join(
        FORENSIC_OUTPUT_PATH,
        "forensic_missing_value_analysis.csv"
    ),
    index=False
)

print("Saved: forensic_missing_value_analysis.csv")

Saved: forensic_missing_value_analysis.csv


In [ ]:
fraud_distribution = (
    df["fraud_type"]
    .value_counts()
    .reset_index()
)

fraud_distribution.columns = [
    "fraud_type",
    "transaction_count"
]

display(fraud_distribution)

,fraud_type,transaction_count
0,Normal,148544
1,Mule,300
2,Structuring,300
3,Smurfing,250
4,Circular,200
5,RapidGeographicMovement,200
6,Layering,200


In [ ]:
amount_stats = df["amount"].describe()

display(amount_stats)

,amount
count,149994.000000
mean,24103.119973
std,78323.420185
min,100.000000
25%,3023.000000
50%,7430.000000
75%,19272.750000
max,999431.000000


## 1. Data Profiling

Data profiling provides an initial understanding of the transaction population.

We examine:

- Dataset dimensions
- Missing values
- Transaction types
- Fraud distribution
- Amount statistics
- Risk categories
- Timestamp coverage

This establishes the baseline against which forensic anomalies can be interpreted.